# Local vs global model #
Material for the choice between one global model and one model per price area. The CNN in `neural_network.ipynb` is global: one network for all six locations, with location as a one-hot input. Here its trained convolutions are frozen and a new dense head (same layers as the CNN's head) is trained from scratch for each location on that location's rows only. The two are compared on the same test hours as the other models.

* **Global CNN**: the networks saved by `neural_network.ipynb`, unchanged.
* **Local heads**: the same convolutions, plus one head per location and seed network. The number of epochs per location is chosen by early stopping on the three validation windows; everything else (loss, optimizer, learning rate, batch size, dropout) is the global CNN's.

Requires `results/neural_network_models.pt`, which `neural_network.ipynb` writes. Takes a few minutes, as the heads train on the cached output of the convolutions.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from forecast_checks import LeakageChecker, ForecastValidator, MakeTrainingData, paired_bootstrap
from neural_network_helper import SEED, STATIC_COLS, WindowScaler, load_ensemble, ensemble_predict
from local_heads_helper import head_epochs, fit_local_heads, predict_local_heads

torch.manual_seed(SEED)
np.random.seed(SEED)
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)
AREAS = {"oslo": "NO1 Oslo", "stavanger": "NO2 Stavanger", "trondheim": "NO3 Trondheim", "tromsø": "NO4 Tromsø",
         "bergen": "NO5 Bergen", "helsingfors": "FI Helsingfors"}

In [ ]:
#Same data, split, validation windows and test hours as neural_network.ipynb, so the saved networks see the same inputs
WINDOW_HOURS = 168
trainer = MakeTrainingData(window_hours=WINDOW_HOURS)
df = trainer.df
data = trainer.get_train_test()
train, test = (df.loc[data[part]["feature"].index] for part in ("train", "test"))
LeakageChecker.check_split(train["date"], test["date"])
folds = [(train.loc[fit], train.loc[val]) for fit, val in MakeTrainingData.validation_windows(train["date"])]

feats = ["location", "hour", "weekday", *STATIC_COLS, *trainer.consumption_window, *trainer.temperature_window]
LeakageChecker.check_feature_names(feats)
LOCATIONS = sorted(df["location"].unique())
scored = feats + ["consumption_6days_ago", "consumption_7days_ago", "consumption_14days_ago"]
train, test = (d.dropna(subset=scored) for d in (train, test))
folds = [(fit.dropna(subset=scored), val.dropna(subset=scored)) for fit, val in folds]


def make_scaler(data):
    return WindowScaler(data, trainer.consumption_window, trainer.temperature_window, LOCATIONS)

In [ ]:
#The networks of neural_network.ipynb: the ensemble refitted on the whole training period, and one per validation window
saved = torch.load("results/neural_network_models.pt")
params = saved["params"]
scaler = make_scaler(train)
n_static = scaler.X(test.iloc[:1])[1].shape[1]
global_models = load_ensemble(saved["test"], params, WINDOW_HOURS, n_static)
fold_models = [load_ensemble(states, params, WINDOW_HOURS, n_static) for states in saved["folds"]]

#The loaded networks must give exactly the forecasts neural_network.ipynb saved, otherwise the inputs or scaling differ
global_pred = ensemble_predict(global_models, test, scaler)
saved_cnn = ForecastValidator.load_predictions("results/neural_network_predictions.json")
assert saved_cnn.actual.index.equals(test.index), "Not the test hours of neural_network.ipynb"
assert np.allclose(global_pred, saved_cnn.forecasts["Neural network"]), "Loaded networks do not reproduce the saved forecasts"
print(f"{len(global_models)} seed networks, {len(fold_models)} validation windows; params {params}")

In [ ]:
#Number of epochs per location: early stopping of each head on each validation window, for each seed network.
#The test period is not used. The mean over windows and networks is used for the final heads
start = pd.Timestamp.now()
stopping = pd.concat([head_epochs(networks, fit, val, make_scaler(fit), params).assign(window=k)
                      for k, (networks, (fit, val)) in enumerate(zip(fold_models, folds))])
epochs = stopping.groupby("location")["epoch"].mean().round().astype(int).to_dict()
print(f"{(pd.Timestamp.now() - start).total_seconds():.0f} s; epochs per location: {epochs}")
stopping.pivot_table(index="location", columns="window", values="epoch", aggfunc="mean")

In [ ]:
#Validation forecasts with those epochs, like the global CNN's in neural_network.ipynb (fitted up to 6 days before each
#window). The windows contain Christmas and New Year, which the test period does not. Note that the epochs were chosen
#on these windows, so the local heads' validation scores are slightly optimistic
val_global, val_local = [], []
for networks, (fit, val) in zip(fold_models, folds):
    fit_scaler = make_scaler(fit)
    val_global.append(ensemble_predict(networks, val, fit_scaler))
    val_local.append(predict_local_heads(networks, fit_local_heads(networks, fit, fit_scaler, params, epochs), val, fit_scaler))
val_all = pd.concat([val for _, val in folds])
val_validator = (ForecastValidator(val_all["consumption"], groups=val_all["location"])
                 .add("Same hour last week", val_all["consumption_7days_ago"], baseline=True)
                 .add("Global CNN", np.concatenate(val_global))
                 .add("Local heads", np.concatenate(val_local)))
val_validator.save_predictions("results/local_vs_global_validation_predictions.json")
val_validator.per_group(skill=True)

In [ ]:
CHRISTMAS = ("2022-12-24", "2023-01-01")
period = (val_all["date"].between(*CHRISTMAS)
          .map({True: "Christmas and New Year (24 Dec - 1 Jan)", False: "Other validation days"}).rename("period"))
val_validator.per(period, skill=True)

In [ ]:
#Heads trained on the whole training period on top of the saved test networks, then the test period is predicted once
heads = fit_local_heads(global_models, train, scaler, params, epochs)
local_pred = predict_local_heads(global_models, heads, test, scaler)
validator = (ForecastValidator(test["consumption"], groups=test["location"])
             .add("Same hour last week", test["consumption_7days_ago"], baseline=True)
             .add("Global CNN", global_pred)
             .add("Local heads", local_pred))
validator.save_predictions("results/local_vs_global_predictions.json")
validator.summary(skill=True)

In [ ]:
validator.per_group(skill=True)

## Local heads relative to the global CNN ##
Skill = 1 - MAE of the local heads / MAE of the global CNN, per price area: above 0 means the local heads are better. The interval is a 95 % bootstrap interval of the MAE difference (local - global) over whole days, so a difference is only counted as real when the interval excludes 0.

In [ ]:
vs_global = (ForecastValidator(test["consumption"], groups=test["location"].astype(str).map(AREAS).rename("Price area"))
             .add("Global CNN", global_pred, baseline=True)
             .add("Local heads", local_pred))
skill_vs_global = 100 * vs_global.per_group(skill=True).loc["Local heads", "skill"].reindex(AREAS.values())

#Per area and for all areas together
bootstrap = pd.concat([
    paired_bootstrap(vs_global.actual, vs_global.forecasts["Local heads"], vs_global.forecasts["Global CNN"],
                     test["date"], vs_global.groups).reindex(AREAS.values()),
    paired_bootstrap(vs_global.actual, vs_global.forecasts["Local heads"], vs_global.forecasts["Global CNN"],
                     test["date"], pd.Series("All", index=test.index, name="Price area"))])
overall_skill = 100 * vs_global.summary(skill=True).loc["Local heads", "skill"]
bootstrap.assign(skill_vs_global=pd.concat([skill_vs_global, pd.Series({"All": overall_skill})])).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5), layout="constrained")
skill_vs_global[::-1].plot.barh(ax=ax, color=validator.colors(["Local heads"]))
ax.axvline(0, color="#52514e", linewidth=0.8)
ax.set_xlabel("Skill of local heads vs global CNN [%]")
ax.set_ylabel("")
fig.savefig(FIGURES / "local_vs_global.pdf")

In [ ]:
#Seed-to-seed spread: the test MAE of each seed network alone. A difference between the two variants that is smaller
#than this spread is within the noise of training
per_seed = pd.DataFrame({
    "Global CNN": [ForecastValidator.score(test["consumption"], ensemble_predict([model], test, scaler))["mae"]
                   for model in global_models],
    "Local heads": [ForecastValidator.score(test["consumption"], predict_local_heads([model], [head], test, scaler))["mae"]
                    for model, head in zip(global_models, heads)]}).rename_axis("seed network")
per_seed.describe().loc[["mean", "std", "min", "max"]]

## Conclusion (fill in after running) ##
Points for the report section *Models: one global model vs. one model per area*:

* **Accuracy per area**: in which areas do the local heads beat the global CNN, and is the difference significant (bootstrap interval) and larger than the seed spread? Helsingfors is the area to watch: the global CNN loses to the baseline there.
* **Data per location**: each head learns from fewer than 6,000 training hours (Helsingfors fewer, as its series starts in July), against about 36,000 for the global model, so local models are more prone to overfitting.
* **Holidays**: Christmas and New Year happen once in the data; compare the two on the validation windows.
* **Maintenance**: one model against six heads to retrain, monitor and version.
* **New areas or retailers**: a shared trunk can serve a new area with little history by training only a small head.
* **Portfolio level shifts** are per location; a local head can be retrained for one area without touching the others.